# Gold Layer — agg_restaurant_ranking (design 1.5, split credit)

One row per restaurant (`restaurant_id`, current version for name / address). **Full overwrite** daily (`CREATE OR REPLACE`, atomic): the score is non-additive, so it is always recomputed. Run **after** `Analytic/gold/checks`.

| Column | Definition |
| --- | --- |
| `video_count` | `SUM(mention_count)` |
| `credited_views` | `SUM(latest view_count × mention_weight)` (split credit: a "top 10 quán" video gives each 1/10 of its views) |
| `credited_views_30d` | `SUM(views_gained in the last 30 days × mention_weight)` |
| `like_rate` | `SUM(likes × weight) ÷ SUM(views × weight)` (never an average of per-video rates) |
| `ranking_score` | `0.7 × credited_views_30d + 0.3 × like_rate`, each min-max scaled 0–1 within the city (first version, tune later) |
| `rank_in_city` | rank by `ranking_score` within the city |

"Last 30 days" is relative to the latest snapshot day, so a rerun gives the same result.


In [ ]:
# --- Config ---
# --- Layer schemas: job parameters bronze / silver / gold (catalog.schema), default prod.* ---
for layer in ("bronze", "silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
BRONZE, SILVER, GOLD = (dbutils.widgets.get(layer) for layer in ("bronze", "silver", "gold"))

AGG = f"{GOLD}.agg_restaurant_ranking"
WINDOW_DAYS = 30
W_VIEWS_30D, W_LIKE_RATE = 0.7, 0.3

print(f"Gold source: {GOLD}.fact_video_restaurant_mention, fact_video_daily_snapshot, dim_restaurant")
print(f"Gold target: {AGG}")


In [ ]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {AGG}
    COMMENT 'Restaurant ranking (split credit), full overwrite daily. Non-additive: never sum ranking_score.'
    AS
    WITH latest_day AS (
        SELECT MAX(snapshot_date_key) AS k,
               CAST(date_format(date_sub(to_date(CAST(MAX(snapshot_date_key) AS STRING), 'yyyyMMdd'), {WINDOW_DAYS - 1}), 'yyyyMMdd') AS INT) AS window_start
        FROM {GOLD}.fact_video_daily_snapshot
    ),
    video_latest AS (   -- semi-additive counts: take each video's last snapshot
        SELECT video_key, view_count, like_count
        FROM (SELECT *, ROW_NUMBER() OVER (PARTITION BY video_key ORDER BY snapshot_date_key DESC) AS rn
              FROM {GOLD}.fact_video_daily_snapshot)
        WHERE rn = 1
    ),
    video_30d AS (      -- additive gains: sum over the window
        SELECT f.video_key, SUM(COALESCE(f.views_gained, 0)) AS views_gained_30d
        FROM {GOLD}.fact_video_daily_snapshot f CROSS JOIN latest_day l
        WHERE f.snapshot_date_key BETWEEN l.window_start AND l.k
        GROUP BY f.video_key
    ),
    credited AS (
        SELECT d.restaurant_id,
               SUM(m.mention_count)                                  AS video_count,
               SUM(COALESCE(vl.view_count, 0) * m.mention_weight)    AS credited_views,
               SUM(COALESCE(v30.views_gained_30d, 0) * m.mention_weight) AS credited_views_30d,
               SUM(COALESCE(vl.like_count, 0) * m.mention_weight)    AS credited_likes,
               SUM(CASE WHEN vl.like_count IS NOT NULL THEN COALESCE(vl.view_count, 0) * m.mention_weight END) AS credited_views_with_likes
        FROM {GOLD}.fact_video_restaurant_mention m
        JOIN {GOLD}.dim_restaurant d ON d.restaurant_key = m.restaurant_key
        LEFT JOIN video_latest vl ON vl.video_key = m.video_key
        LEFT JOIN video_30d v30   ON v30.video_key = m.video_key
        WHERE m.restaurant_key <> -1
        GROUP BY d.restaurant_id
    ),
    enriched AS (
        SELECT c.*, cur.restaurant_name, cur.address, cur.district, COALESCE(cur.city, 'Unknown') AS city,
               c.credited_likes / NULLIF(c.credited_views_with_likes, 0) AS like_rate
        FROM credited c
        JOIN {GOLD}.dim_restaurant cur ON cur.restaurant_id = c.restaurant_id AND cur.is_current
    ),
    scaled AS (
        SELECT *,
            COALESCE((credited_views_30d - MIN(credited_views_30d) OVER w)
                     / NULLIF(MAX(credited_views_30d) OVER w - MIN(credited_views_30d) OVER w, 0), 0) AS views_30d_scaled,
            COALESCE((like_rate - MIN(like_rate) OVER w)
                     / NULLIF(MAX(like_rate) OVER w - MIN(like_rate) OVER w, 0), 0) AS like_rate_scaled
        FROM enriched
        WINDOW w AS (PARTITION BY city)
    )
    SELECT
        restaurant_id, restaurant_name, address, district, city,
        CAST(video_count AS INT)                  AS video_count,
        CAST(round(credited_views) AS BIGINT)     AS credited_views,
        CAST(round(credited_views_30d) AS BIGINT) AS credited_views_30d,
        like_rate,
        {W_VIEWS_30D} * views_30d_scaled + {W_LIKE_RATE} * like_rate_scaled AS ranking_score,
        CAST(RANK() OVER (PARTITION BY city ORDER BY {W_VIEWS_30D} * views_30d_scaled + {W_LIKE_RATE} * like_rate_scaled DESC,
                                                     credited_views DESC) AS INT) AS rank_in_city,
        (SELECT k FROM latest_day)                AS as_of_date_key,
        current_timestamp()                       AS etl_date
    FROM scaled
""")


In [ ]:
# --- Verify ---
s = spark.sql(f"""
    SELECT COUNT(*) AS n, COUNT(DISTINCT restaurant_id) AS ids, COUNT(DISTINCT city) AS cities, MAX(as_of_date_key) AS as_of
    FROM {AGG}
""").collect()[0]
print(f"📊 {AGG}: {s['n']} restaurants ({s['ids']} distinct ids) in {s['cities']} city(ies), as of {s['as_of']} "
      + ("✅" if s["n"] == s["ids"] else "❌ duplicates"))
display(spark.sql(f"SELECT rank_in_city, restaurant_name, district, video_count, credited_views, credited_views_30d, round(like_rate, 4) AS like_rate, round(ranking_score, 3) AS score FROM {AGG} ORDER BY city, rank_in_city LIMIT 20"))
print("\n--- agg_restaurant_ranking complete ---")
